In [1]:
import sys
import site

print("Python executable:", sys.executable)
print("User site enabled:", site.ENABLE_USER_SITE)
print("sys.path first entries:")
for p in sys.path[:8]:
    print(p)

Python executable: /home/fjh3941/.conda/envs/cv_torch/bin/python
User site enabled: False
sys.path first entries:
/home/fjh3941/.conda/envs/cv_torch/lib/python311.zip
/home/fjh3941/.conda/envs/cv_torch/lib/python3.11
/home/fjh3941/.conda/envs/cv_torch/lib/python3.11/lib-dynload

/home/fjh3941/.conda/envs/cv_torch/lib/python3.11/site-packages


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA build version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))
    x = torch.randn(3, 3).cuda()
    print("Tensor is on:", x.device)
else:
    print("CUDA is not available in this environment/kernel.")

PyTorch version: 2.10.0
CUDA build version: 12.9
CUDA available: True
GPU count: 1
Current GPU: NVIDIA H100 80GB HBM3
Tensor is on: cuda:0


In [7]:
import os
import math
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torchvision.utils import save_image
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import Subset
from tqdm import tqdm
from datasets import load_from_disk

In [9]:
# -------------------------
# Config
# -------------------------

DATA_DIR = "processed_animals_256"

IMG_SIZE = 64
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-4
TIMESTEPS = 100
DEBUG_NUM_IMAGES = 32

# IMG_SIZE = 256
# BATCH_SIZE = 8
# EPOCHS = 50
# LR = 2e-4
# TIMESTEPS = 200
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("samples", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)


# -------------------------
# Dataset
# -------------------------

class AnimalDataset(Dataset):
    def __init__(self, data_dir):
        dataset_dict = load_from_disk(data_dir)

        if "train" in dataset_dict:
            self.dataset = dataset_dict["train"]
        else:
            self.dataset = dataset_dict

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image = self.dataset[idx]["pixel_values"]

        image = torch.tensor(image, dtype=torch.float32)

        # Convert H x W x C into C x H x W
        image = image.permute(2, 0, 1)

        # Normalize [0, 255] to [-1, 1]
        image = image / 127.5 - 1.0

        return image

dataset = AnimalDataset(DATA_DIR)

debug_dataset = Subset(dataset, range(min(DEBUG_NUM_IMAGES, len(dataset))))

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

FileNotFoundError: Directory processed_animals_256 not found

In [ ]:

# -------------------------
# Offset Cosine Diffusion Schedule
# -------------------------

def offset_cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)

    alpha_hats = torch.cos(
        ((x / timesteps) + s) / (1 + s) * math.pi * 0.5
    ) ** 2

    alpha_hats = alpha_hats / alpha_hats[0]

    betas = 1 - (alpha_hats[1:] / alpha_hats[:-1])
    betas = torch.clip(betas, 1e-4, 0.999)

    return betas

betas = offset_cosine_beta_schedule(TIMESTEPS).to(DEVICE)
alphas = 1.0 - betas
alpha_hats = torch.cumprod(alphas, dim=0)


def add_noise(x, t):
    noise = torch.randn_like(x)

    alpha_hat = alpha_hats[t].view(-1, 1, 1, 1)

    noisy_x = torch.sqrt(alpha_hat) * x + torch.sqrt(1 - alpha_hat) * noise

    return noisy_x, noise


# -------------------------
# Time Embedding
# -------------------------

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb_scale = math.log(10000) / (half_dim - 1)

        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb_scale)
        emb = t[:, None] * emb[None, :]

        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)


# -------------------------
# U-Net Blocks
# -------------------------

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.norm1 = nn.GroupNorm(8, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)

        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_emb = self.time_mlp(t)
        h = h + time_emb[:, :, None, None]

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.skip(x)


class UNet(nn.Module):
    def __init__(self, img_channels=3, base_channels=64, time_dim=256):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.enc1 = ResBlock(img_channels, base_channels, time_dim)
        self.enc2 = ResBlock(base_channels, base_channels * 2, time_dim)
        self.enc3 = ResBlock(base_channels * 2, base_channels * 4, time_dim)
        self.enc4 = ResBlock(base_channels * 4, base_channels * 8, time_dim)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ResBlock(base_channels * 8, base_channels * 8, time_dim)

        self.up4 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec4 = ResBlock(base_channels * 8, base_channels * 4, time_dim)

        self.up3 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec3 = ResBlock(base_channels * 4, base_channels * 2, time_dim)

        self.up2 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec2 = ResBlock(base_channels * 2, base_channels, time_dim)

        self.up1 = nn.ConvTranspose2d(base_channels, base_channels, 2, stride=2)
        self.dec1 = ResBlock(base_channels * 2, base_channels, time_dim)

        self.out = nn.Conv2d(base_channels, img_channels, 1)

    def forward(self, x, t):
        t = self.time_embedding(t)

        e1 = self.enc1(x, t)
        e2 = self.enc2(self.pool(e1), t)
        e3 = self.enc3(self.pool(e2), t)
        e4 = self.enc4(self.pool(e3), t)

        b = self.bottleneck(self.pool(e4), t)

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4, t)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3, t)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2, t)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1, t)

        return self.out(d1)


# -------------------------
# Sampling
# -------------------------

@torch.no_grad()
def sample(model, n=4):
    model.eval()

    x = torch.randn(n, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

    for i in tqdm(reversed(range(TIMESTEPS)), desc="Sampling"):
        t = torch.full((n,), i, device=DEVICE, dtype=torch.long)

        predicted_noise = model(x, t)

        alpha = alphas[t].view(-1, 1, 1, 1)
        alpha_hat = alpha_hats[t].view(-1, 1, 1, 1)
        beta = betas[t].view(-1, 1, 1, 1)

        if i > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (1 / torch.sqrt(alpha)) * (
            x - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise
        ) + torch.sqrt(beta) * noise

    x = torch.clamp(x, -1, 1)
    x = (x + 1) / 2

    return x


FileNotFoundError: [Errno 2] No such file or directory: 'processed_animals_256/train'

Training

In [ ]:
# -------------------------
# Training
# -------------------------

model = UNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    loop = tqdm(loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for images, _ in loop:
        images = images.to(DEVICE)

        t = torch.randint(0, TIMESTEPS, (images.size(0),), device=DEVICE).long()

        noisy_images, noise = add_noise(images, t)

        predicted_noise = model(noisy_images, t)

        loss = F.mse_loss(predicted_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch + 1}: Loss = {avg_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        generated = sample(model, n=4)
        save_image(generated, f"samples/sample_epoch_{epoch + 1}.png", nrow=2)
        torch.save(model.state_dict(), f"checkpoints/ddpm_epoch_{epoch + 1}.pt")

torch.save(model.state_dict(), "checkpoints/ddpm_final.pt")